In [11]:
from vllm import AsyncLLMEngine, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
import json



from pyngrok import ngrok
import nest_asyncio
import asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

In [2]:
engine_args = AsyncEngineArgs(model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", tensor_parallel_size=8, max_model_len=4096, gpu_memory_utilization=0.3)

In [3]:
engine = AsyncLLMEngine.from_engine_args(engine_args)

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-06-03 12:45:34,001	INFO worker.py:1749 -- Started a local Ray instance.


INFO 06-03 12:45:35 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 06-03 12:45:59 utils.py:618] Found nccl from library libnccl.so.2
INFO 06-03 12:45:59 pynccl.py:65] vLLM is using nccl==2.20.5
(RayWorkerWrapper pid=438822) INFO 06-03 12:45:59 utils.py:618] Found nccl from library libnccl.so.2
(RayWorkerWrap

In [13]:
sampling_params = SamplingParams(temperature=0.3, top_p=0.95, top_k=50, max_tokens=4096)

In [14]:

# 입력값 변수
talent = """
1. 끊임없는 열정으로 미래에 도전하는 인재
2. 창의와 혁신으로 세상을 변화시키는 인재
3. 정직과 바른 행동으로 역할과 책임을 다하는 인재
"""
user_answer = """
[[기계공학&화학 지식 동시 겸비]
저는 기계공학과 주전공, 화학 부전공, 고1 전국 화학올림피아드 금상을 타며 기계공학&화학 지식을 모두 겸비하고 있습니다.
저는 기계공학과에서 4대역학과 열/응력 시뮬레이션을 가장 흥미있게 공부했습니다. FEM방식을 이용, matlab 코딩, 고정보에 균일하중이 가해질 때 stress&strain distribution을 simulation을 해봤습니다. 와이어나 범프에서 열이 발생하는 harsh condition에서, 패키지가 응력/열적으로 안정성을 갖추는지 simulation 해보는 업무에 도움이 될 것입니다. 4모터 수동로봇을 solidworks로 3D 설계/simulation/제작 해 본 경험, 열전달 실험에서 Fin의 열전달 효율을 계산해 본 경험, 유체역학실험에서 베르누이 방정식으로 airfoil의 양력/항력/박리점 계산을 해 본 경험도 있습니다.
저는 화학과에서 STM, RAMAN, electroosmosis, cyclovoltammetry등 다양한 실험 장비를 다뤄봤고, 분석을 해봤습니다.
반도체 전공정은 이미 3nm 양산을 앞두고 있습니다. 앞으로의 반도체 전쟁은 후공정에 달려있다고 합니다. 제 기계공학&화학 지식을 적극 활용할 수 있는 TSP 패키지 개발 팀에서, 삼성전자의 패키지 기술을 international top tier로 만드는 1류 엔지니어가 되고 싶습니다.]
"""

# 프롬프트
prompt = f"""당신은 인재상을 기준으로 사용자의 자기소개서를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정을 지키면서 작업을 수행하세요.

[입력 정보]
인재상: {talent}
자기소개서: {user_answer}

[첨삭 과정]
1. 인재상을 확인하세요.
2. 제공된 자기소개서의 내용과 흐름을 분석하세요.
3. 자기소개서에서 지원자의 인성 및 역량 중 인재상과 부합하는 부분을 식별하세요.
4. 식별된 부분을 강조하고, 해당 문단에서 인재상과 관련된 경험이나 역량을 구체적으로 설명하세요.
5. 모든 문단에 걸쳐 인재상이 고르게 반영되도록 하세요. 
6. 결론은 제외하세요.
7. 출력형식은 다음을 지키세요.

[첨삭]
"""
formatted_prompt = f"instruction: {prompt}\n output:"

example_input = {
"prompt": formatted_prompt,
"stream": False, # assume the non-streaming case
"request_id": talent,
}

# 비동기 추론
results_generator = engine.generate(
    example_input["prompt"],
    sampling_params,
    example_input["request_id"]
)

final_output = None



In [ ]:
final_output.outputs[0].text